# IMDB Sentiment Analysis - Mini Project
**Project:** Impact of Preprocessing Choices on Classical NLP Sentiment Analysis  
**Team Members:** Mudassir Iftikhar & Keshav Gupta

---

## What are we doing here?

We want to figure out whether a movie review is **positive** or **negative** - that's sentiment analysis.  
The dataset is the famous **IMDB Movie Reviews** dataset (50,000 reviews, perfectly balanced).

The big question we're exploring: **does cleaning the text better actually make the model more accurate?**  
We'll test 5 different preprocessing setups and compare results using **Logistic Regression**.

### Notebook Flow
1. Import libraries  
2. Download NLTK resources  
3. Load the dataset  
4. Explore the data (EDA)  
5. Define preprocessing functions  
6. Apply preprocessing and inspect each step  
7. Split into train/test  
8. Run 5 experiments with Logistic Regression  
9. Compare results with plots  
10. Analyse the best-performing experiment

## Step 1 - Import Libraries

We bring in everything we need up front.  
- `pandas` / `numpy` - data handling  
- `nltk` - tokenizing and stopword lists  
- `sklearn` - TF-IDF vectorizer, Logistic Regression, and evaluation metrics  
- `matplotlib` / `seaborn` - charts  

We also set a few **global constants** (random seed, test split size, vocab size, file path) so they're easy to change in one place.

In [ ]:
import re
import string
import time
import warnings

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)

# ── Global settings ────────────────────────────────────────────────────────────
RANDOM_STATE = 42          # keeps results reproducible every run
TEST_SIZE    = 0.20        # 20% of data goes to testing, 80% to training
MAX_FEATURES = 10_000      # TF-IDF will only keep the top 10,000 words
DATASET_PATH = "IMDB.csv"  # make sure this file is in the same folder

print("All libraries imported — ready to go!")

: 

## Step 2 - Download NLTK Resources

NLTK needs two things we'll use later:
- **punkt** - a pre-trained sentence/word tokenizer  
- **stopwords** - a list of common English words like *the*, *is*, *and* that carry little meaning

The code checks if they're already downloaded so it doesn't re-download every time.

In [ ]:
# List of (internal path, download name) pairs for the resources we need
required_resources = [
    ("tokenizers/punkt",     "punkt"),
    ("tokenizers/punkt_tab", "punkt_tab"),
    ("corpora/stopwords",    "stopwords"),
]

for path, name in required_resources:
    try:
        nltk.data.find(path)           # if found, do nothing
    except LookupError:
        print(f"Downloading {name}...")  # if missing, download it
        nltk.download(name, quiet=True)

# Build the stopword set and stemmer once so they're reused everywhere
STOP_WORDS = set(stopwords.words("english"))
STEMMER    = PorterStemmer()

print("NLTK resources are ready.")

## Step 3 - Load the Dataset

We load `IMDB.csv` and normalise the column names to just `text` and `label`.  
The label column might say *positive/negative* or *1/0* - we convert everything to **1 = Positive, 0 = Negative**.

In [ ]:
df = pd.read_csv(DATASET_PATH)

# Standardise column names (strip spaces, make lowercase)
df.columns = [col.strip().lower() for col in df.columns]

# Find whichever column holds the review text and whichever holds the label
text_col  = next((c for c in df.columns if c in ("review", "text", "content")), None)
label_col = next((c for c in df.columns if c in ("sentiment", "label", "class")),  None)

if text_col is None or label_col is None:
    raise ValueError(f"Couldn't find expected columns. Got: {list(df.columns)}")

df = df.rename(columns={text_col: "text", label_col: "label"})
df = df[["text", "label"]].dropna().reset_index(drop=True)
df["text"] = df["text"].astype(str)

# Convert text labels (positive/negative) to numbers (1/0) if needed
if not pd.api.types.is_numeric_dtype(df["label"]):
    label_map = {"positive": 1, "pos": 1, "negative": 0, "neg": 0}
    mapped = df["label"].str.strip().str.lower().map(label_map)
    df["label"] = mapped if mapped.notna().all() else pd.Categorical(df["label"]).codes

df["label"] = df["label"].astype(int)

print(f"Dataset loaded - {df.shape[0]:,} rows, {df.shape[1]} columns")
display(df.head(3))

## Step 4 - Exploratory Data Analysis (EDA)

Before jumping into modelling, it's good practice to understand the data.  
We check for **missing values**, **duplicates**, and look at the **review length distribution**.

In [ ]:
# ── Basic data quality checks ──────────────────────────────────────────────────
print(f"Total reviews  : {len(df):,}")
print(f"Missing (text) : {df['text'].isna().sum()}")
print(f"Missing (label): {df['label'].isna().sum()}")
print(f"Duplicates     : {df.duplicated().sum()}")

# ── Review length distribution ─────────────────────────────────────────────────
# character count gives a quick sense of how long the reviews are
df["text_length"] = df["text"].str.len()
display(df[["text_length"]].describe().T)

plt.figure(figsize=(8, 4))
sns.histplot(df["text_length"], bins=40, kde=True, color="steelblue")
plt.title("Distribution of Review Lengths (characters)")
plt.xlabel("Number of characters")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## Step 5 - Class Balance Check

A balanced dataset (equal positives and negatives) means our model won't be biased towards one class.  
IMDB is well-known for being 50/50 - let's confirm that.

In [ ]:
label_names   = {0: "Negative", 1: "Positive"}
label_series  = df["label"].map(label_names)
label_counts  = label_series.value_counts().reindex(["Negative", "Positive"], fill_value=0)

display(label_counts.to_frame(name="count"))

plt.figure(figsize=(6, 4))
ax = sns.countplot(x=label_series, order=["Negative", "Positive"],
                   palette=["#e76f51", "#2a9d8f"])
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")

# add count labels on top of each bar
for bar in ax.patches:
    h = int(bar.get_height())
    ax.annotate(str(h), (bar.get_x() + bar.get_width() / 2, h),
                ha="center", va="bottom")

plt.tight_layout()
plt.show()

print("Class split (%):")
display((label_counts / len(df) * 100).round(2).to_frame(name="percentage"))

## Step 6 — Text Preprocessing Functions

Raw text is messy - it contains HTML tags, punctuation, URLs, numbers, and filler words that don't help the model.  
We define small, focused functions for each cleaning step so we can mix and match them in our experiments.

| Function | What it does |
|---|---|
| `lowercase_text` | Makes everything lowercase so *Good* and *good* are treated the same |
| `remove_punctuation` | Strips commas, dots, exclamation marks, etc. |
| `tokenize_text` | Splits a sentence into individual words (tokens) |
| `remove_noise` | Removes URLs, HTML tags, and numbers |
| `remove_special_symbols` | Keeps only purely alphabetic words |
| `stem_tokens` | Cuts words to their root form (*running* → *run*) |
| `remove_stop_words` | Drops common words like *the*, *is*, *a* that don't carry sentiment |
| `join_tokens` | Glues the token list back into a single string |

In [ ]:
# ── Individual cleaning functions ──────────────────────────────────────────────

def lowercase_text(text):
    # convert to string first in case of any non-string entries, then lowercase
    return str(text).lower()


def remove_punctuation(text):
    # str.maketrans creates a mapping that deletes all punctuation characters
    return text.translate(str.maketrans("", "", string.punctuation))


def tokenize_text(text):
    # splits "this is great" → ["this", "is", "great"]
    return nltk.word_tokenize(text)


def remove_noise(tokens):
    # remove URLs, HTML tags, and numeric tokens from the word list
    cleaned = []
    for token in tokens:
        token = re.sub(r"http\S+|www\S+", "", token)   # strip URLs
        token = re.sub(r"<.*?>",           "", token)   # strip HTML tags
        token = re.sub(r"\d+",             "", token)   # strip digits
        token = token.strip()
        if token:                                        # skip empty strings
            cleaned.append(token)
    return cleaned


def remove_special_symbols(tokens):
    # only keep tokens that are made up entirely of letters (a-z, A-Z)
    return [t for t in tokens if re.fullmatch(r"[a-zA-Z]+", t)]


def stem_tokens(tokens):
    # Porter stemmer reduces words to their base form
    # e.g. "movies" → "movi", "running" → "run"
    return [STEMMER.stem(t) for t in tokens]


def remove_stop_words(tokens):
    # drop words that appear in the English stopword list
    return [t for t in tokens if t not in STOP_WORDS]


def join_tokens(tokens):
    # TF-IDF expects a string, not a list — so we join tokens back
    return " ".join(tokens)


print("Preprocessing functions defined.")

## Step 7 - Build the Experiment Pipelines

Each experiment uses a **different combination** of the functions above.  
Keeping them as separate functions makes it easy to swap in different preprocessing in the experiments later.

In [ ]:
# ── Preprocessing pipelines (one per experiment) ───────────────────────────────

def preprocess_basic(texts):
    """Exp1 - only lowercase, remove punctuation, tokenize.
    This is our baseline — minimal cleaning, no stopword removal, no stemming."""
    s = pd.Series(texts).apply(lowercase_text).apply(remove_punctuation)
    return s.apply(tokenize_text).apply(join_tokens)


def preprocess_stopwords(texts):
    """Exp2 - basic + remove stopwords.
    Removes filler words so the model focuses on meaningful words."""
    s = pd.Series(texts).apply(lowercase_text).apply(remove_punctuation)
    tokens = s.apply(tokenize_text).apply(remove_stop_words)
    return tokens.apply(join_tokens)


def preprocess_stemming(texts):
    """Exp3 - basic + stemming.
    Reduces words to their root so 'loving' and 'loved' are treated the same."""
    s = pd.Series(texts).apply(lowercase_text).apply(remove_punctuation)
    tokens = s.apply(tokenize_text).apply(stem_tokens)
    return tokens.apply(join_tokens)


def preprocess_stopwords_stemming(texts):
    """Exp4 - basic + remove stopwords + stemming.
    Combined effect: cleaner vocabulary and reduced word forms."""
    s = pd.Series(texts).apply(lowercase_text).apply(remove_punctuation)
    tokens = s.apply(tokenize_text).apply(remove_stop_words).apply(stem_tokens)
    return tokens.apply(join_tokens)


def preprocess_stopwords_bigrams(texts):
    """Exp5 - basic + remove stopwords (bigrams handled at TF-IDF level).
    Bigrams let the model pick up two-word phrases like 'not good'."""
    s = pd.Series(texts).apply(lowercase_text).apply(remove_punctuation)
    tokens = s.apply(tokenize_text).apply(remove_stop_words)
    return tokens.apply(join_tokens)


print("Pipeline functions ready.")

## Step 8 - Preview Each Preprocessing Step

Before running the full experiments, it's helpful to see exactly what happens to a review at each stage.  
The table below shows the first few reviews after each transformation - great for catching bugs early and understanding the pipeline.

In [ ]:
# Work on a copy so we don't modify the original dataframe
data = df.copy()

# Apply each step in sequence and save as a new column
data["step1_lowercase"]    = data["text"].apply(lowercase_text)
data["step2_no_punct"]     = data["step1_lowercase"].apply(remove_punctuation)
data["step3_tokens"]       = data["step2_no_punct"].apply(tokenize_text)
data["step4_no_noise"]     = data["step3_tokens"].apply(remove_noise)
data["step5_alpha_only"]   = data["step4_no_noise"].apply(remove_special_symbols)
data["step6_stemmed"]      = data["step5_alpha_only"].apply(stem_tokens)
data["step7_no_stopwords"] = data["step6_stemmed"].apply(remove_stop_words)
data["clean_text"]         = data["step7_no_stopwords"].apply(join_tokens)

# Show a small preview table — only the first 3 rows to keep it readable
preview_cols = [
    "text", "step1_lowercase", "step2_no_punct",
    "step3_tokens", "step4_no_noise", "step5_alpha_only",
    "step6_stemmed", "step7_no_stopwords", "clean_text"
]
display(data[preview_cols].head(3))

## Step 9 - Train / Test Split

We split the data: **80% for training** the model, **20% for testing** how well it generalises.  
`stratify=y` ensures both splits have the same 50/50 class ratio - important for a fair evaluation.

In [ ]:
# We split on the *raw* text — each experiment will preprocess it differently
X_raw = data["text"].astype(str)
y     = data["label"].values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,   # keep 50/50 split in both train and test
)

print(f"Training samples : {len(X_train_raw):,}")
print(f"Testing  samples : {len(X_test_raw):,}")

## Step 10 - Helper Functions (Vectorizer + Metrics + Plots)

We define a few helper functions that all five experiments share:

- **`tfidf_vectorize`** - converts text into a numeric matrix using TF-IDF  
- **`get_metrics`** - computes accuracy, precision, recall, and F1 from predictions  
- **`plot_bar_comparison`** - draws a bar chart to compare experiments  
- **`plot_confusion_matrix`** - shows the confusion matrix for a single model

In [ ]:
def tfidf_vectorize(train_texts, test_texts, max_features=10_000, ngram_range=(1, 1)):
    """
    TF-IDF turns text into numbers the model can understand.
    - max_features: only keep the most common N words (reduces noise)
    - ngram_range: (1,1) = unigrams only; (1,2) = unigrams + bigrams
    - sublinear_tf: applies log scaling so very frequent words aren't over-weighted
    """
    vectorizer = TfidfVectorizer(
        max_features=max_features,
        ngram_range=ngram_range,
        sublinear_tf=True,
        strip_accents="unicode",
        token_pattern=r"\b[a-zA-Z]{2,}\b",  # only words with 2+ letters
    )
    # fit on train only — transform both (avoids data leakage)
    X_train_vec = vectorizer.fit_transform(train_texts)
    X_test_vec  = vectorizer.transform(test_texts)
    return X_train_vec, X_test_vec, vectorizer


def get_metrics(y_true, y_pred, label=""):
    """Return a dictionary of the four main classification metrics."""
    return {
        "experiment" : label,
        "accuracy"   : round(accuracy_score(y_true, y_pred),                      4),
        "precision"  : round(precision_score(y_true, y_pred, zero_division=0),    4),
        "recall"     : round(recall_score(y_true, y_pred,    zero_division=0),    4),
        "f1_score"   : round(f1_score(y_true, y_pred,        zero_division=0),    4),
    }


def plot_bar_comparison(results_list, metric="accuracy"):
    """Bar chart comparing a given metric across all experiments."""
    names  = [r["experiment"] for r in results_list]
    values = [r[metric]       for r in results_list]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(names, values, color="steelblue", edgecolor="black", width=0.5)

    # label each bar with its numeric value
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.003,
                f"{val:.4f}", ha="center", va="bottom", fontsize=9)

    ax.set_ylim(0, 1.08)
    ax.set_ylabel(metric.replace("_", " ").title())
    ax.set_title(f"{metric.replace('_', ' ').title()} — All Experiments")
    plt.xticks(rotation=20, ha="right", fontsize=9)
    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(y_true, y_pred, title=""):
    """Plot a labelled confusion matrix (Negative / Positive)."""
    cm   = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Negative", "Positive"])
    _, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("Helper functions defined.")

## Step 11 - Run All 5 Experiments

Here's the heart of the project. We run **Logistic Regression** across five different preprocessing setups:

| Exp | Preprocessing | TF-IDF |
|-----|--------------|--------|
| Exp1 | Lowercase + punctuation removal + tokenization | Unigrams |
| Exp2 | Exp1 + stopword removal | Unigrams |
| Exp3 | Exp1 + stemming | Unigrams |
| Exp4 | Exp1 + stopword removal + stemming | Unigrams |
| Exp5 | Exp1 + stopword removal | Unigrams + Bigrams |

For each experiment: **preprocess → vectorize → train → predict → evaluate**.

In [ ]:
# Each tuple: (display name, preprocessing function, TF-IDF ngram range)
EXPERIMENTS = [
    ("Exp1: Basic Tokenization",         preprocess_basic,              (1, 1)),
    ("Exp2: +Stopword Removal",          preprocess_stopwords,          (1, 1)),
    ("Exp3: +Stemming",                  preprocess_stemming,           (1, 1)),
    ("Exp4: +Stopwords & Stemming",      preprocess_stopwords_stemming, (1, 1)),
    ("Exp5: +Stopwords & Bigrams",       preprocess_stopwords_bigrams,  (1, 2)),
]

all_results  = []   # will store metrics dicts for every experiment
all_preds    = {}   # will store (y_true, y_pred) for later confusion matrices

for exp_name, preprocess_fn, ngram_range in EXPERIMENTS:
    print(f"\n{'─'*60}")
    print(f"  {exp_name}")
    print(f"{'─'*60}")

    # 1. Apply the experiment's preprocessing pipeline to raw text
    X_train_clean = preprocess_fn(X_train_raw)
    X_test_clean  = preprocess_fn(X_test_raw)

    # 2. Convert cleaned text into TF-IDF feature matrix
    X_train_vec, X_test_vec, vectorizer = tfidf_vectorize(
        X_train_clean, X_test_clean,
        max_features=MAX_FEATURES,
        ngram_range=ngram_range
    )
    print(f"  Vocabulary size : {len(vectorizer.vocabulary_):,}")
    print(f"  Feature matrix  : {X_train_vec.shape}")

    # 3. Train Logistic Regression
    # C=1.0 is the regularisation strength — lower C = stronger regularisation
    # lbfgs is a reliable solver that works well for text classification
    t_start = time.time()
    model = LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs",
                               random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_train_vec, y_train)
    elapsed = round(time.time() - t_start, 2)

    # 4. Predict on the test set
    y_pred = model.predict(X_test_vec)

    # 5. Compute and store metrics
    metrics = get_metrics(y_test, y_pred, label=exp_name)
    metrics["train_time_s"] = elapsed
    all_results.append(metrics)
    all_preds[exp_name] = (y_test, y_pred)

    print(f"  Accuracy  : {metrics['accuracy']:.4f}")
    print(f"  F1-Score  : {metrics['f1_score']:.4f}")
    print(f"  Train time: {elapsed}s")

print("\nAll experiments complete!")

## Step 12 - Results Summary Table

Let's put all the results in one table, sorted by F1-Score from best to worst.

In [ ]:
results_df = (
    pd.DataFrame(all_results)
    .sort_values(by=["f1_score", "accuracy"], ascending=False)
    .reset_index(drop=True)
)

print("Results — sorted by F1-Score (best first)")
display(results_df)

## Step 13 - Comparison Plots

Visualising the results makes it much easier to spot which preprocessing strategy helped the most.
We plot all four metrics: **Accuracy**, **F1-Score**, **Precision**, and **Recall**.

In [ ]:
for metric in ["accuracy", "f1_score", "precision", "recall"]:
    plot_bar_comparison(all_results, metric=metric)

## Step 14 - Best Model: Confusion Matrix & Detailed Analysis

A **confusion matrix** breaks down predictions into four categories:
- **True Positive (TP)** - correctly predicted positive  
- **True Negative (TN)** - correctly predicted negative  
- **False Positive (FP)** - predicted positive, actually negative (*model was too optimistic*)  
- **False Negative (FN)** - predicted negative, actually positive (*model missed it*)  

We show this for the best-performing experiment.

In [ ]:
# Pick the best experiment (highest F1, then accuracy as tiebreaker)
best_row  = results_df.iloc[0]
best_name = best_row["experiment"]
y_true, y_pred = all_preds[best_name]

print(f"Best experiment : {best_name}")
print(f"Accuracy   : {best_row['accuracy']:.4f}")
print(f"Precision  : {best_row['precision']:.4f}")
print(f"Recall     : {best_row['recall']:.4f}")
print(f"F1-Score   : {best_row['f1_score']:.4f}")

# Confusion matrix plot
plot_confusion_matrix(y_true, y_pred, title=f"Confusion Matrix — {best_name}")

# Full classification report (per-class breakdown)
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Negative", "Positive"]))

In [ ]:
# ── Deeper error analysis ──────────────────────────────────────────────────────
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

# Specificity: how good are we at catching negatives?
specificity = tn / (tn + fp)

# NPV (Negative Predictive Value): when we predict negative, how often are we right?
npv = tn / (tn + fn)

print("Confusion Matrix Breakdown")
print(f"  True Negatives (TN)  : {tn:,}")
print(f"  False Positives (FP) : {fp:,}  ← predicted positive, actually negative")
print(f"  False Negatives (FN) : {fn:,}  ← predicted negative, actually positive")
print(f"  True Positives (TP)  : {tp:,}")
print(f"\n  Specificity          : {specificity:.4f}")
print(f"  Negative Pred. Value : {npv:.4f}")

# Compare best vs second-best
if len(results_df) > 1:
    second = results_df.iloc[1]
    print(f"\nBest vs Second-Best")
    print(f"  Best   : {best_name}")
    print(f"  Second : {second['experiment']}")
    print(f"  F1  improvement  : {best_row['f1_score']  - second['f1_score']:.4f}")
    print(f"  Acc improvement  : {best_row['accuracy']  - second['accuracy']:.4f}")

## Step 15 - Hyperparameter Tuning: `max_features` × `C`

Our experiments showed that **Exp5** (stopword removal + bigrams) performed best, but all results were very close.
A natural next question is: are we leaving performance on the table by not tuning the model itself?

We test two key knobs:

| Parameter | What it controls | Values to try |
|---|---|---|
| `max_features` | How many top words TF-IDF keeps | 5,000 / 10,000 / 20,000 / 50,000 |
| `C` (Logistic Regression) | Regularisation strength - lower C = stronger penalty on large weights | 0.1 / 0.5 / 1.0 / 5.0 / 10.0 |

We run this grid on **Exp5's pipeline** (the best-performing setup) so we're tuning the best candidate.
That gives us **4 × 5 = 20 combinations** to compare.

In [ ]:
import itertools

# ── Grid values ───────────────────────────────────────────────────────────────
MAX_FEATURES_GRID = [5_000, 10_000, 20_000, 50_000]
C_GRID            = [0.1, 0.5, 1.0, 5.0, 10.0]

# We tune on Exp5's pipeline (best from our experiments)
# Preprocess train and test once — same text for all grid iterations
print("Preprocessing text for tuning (Exp5 pipeline)...")
X_train_clean_tune = preprocess_stopwords_bigrams(X_train_raw)
X_test_clean_tune  = preprocess_stopwords_bigrams(X_test_raw)
print("Done. Starting grid search...\n")

tune_results = []  # store all 20 results

for max_f, c_val in itertools.product(MAX_FEATURES_GRID, C_GRID):

    # 1. Vectorize with this max_features setting
    X_tr_vec, X_te_vec, _ = tfidf_vectorize(
        X_train_clean_tune, X_test_clean_tune,
        max_features=max_f,
        ngram_range=(1, 2)   # bigrams, matching Exp5
    )

    # 2. Train Logistic Regression with this C value
    t0  = time.time()
    clf = LogisticRegression(C=c_val, max_iter=1000, solver="lbfgs",
                              random_state=RANDOM_STATE, n_jobs=-1)
    clf.fit(X_tr_vec, y_train)
    elapsed = round(time.time() - t0, 2)

    # 3. Predict and score
    y_pred_tune = clf.predict(X_te_vec)
    metrics     = get_metrics(y_test, y_pred_tune,
                               label=f"max_f={max_f:,}  C={c_val}")
    metrics["max_features"] = max_f
    metrics["C"]            = c_val
    metrics["train_time_s"] = elapsed
    tune_results.append(metrics)

    print(f"  max_features={max_f:>6,}  C={c_val:<5}  "
          f"Acc={metrics['accuracy']:.4f}  F1={metrics['f1_score']:.4f}  ({elapsed}s)")

print("\nGrid search complete!")

In [ ]:
# ── Results table ─────────────────────────────────────────────────────────────
tune_df = (
    pd.DataFrame(tune_results)
    .sort_values(by=["f1_score", "accuracy"], ascending=False)
    .reset_index(drop=True)
)

print("Top 10 combinations (by F1-Score):")
display(tune_df[["max_features", "C", "accuracy", "precision",
                  "recall", "f1_score", "train_time_s"]].head(10))

# ── Heatmap: F1 score for every (max_features, C) pair ───────────────────────
# pivot the data so rows = max_features, columns = C
pivot = tune_df.pivot(index="max_features", columns="C", values="f1_score")

plt.figure(figsize=(9, 4))
sns.heatmap(
    pivot,
    annot=True,          # show F1 value inside each cell
    fmt=".4f",
    cmap="YlGnBu",       # light = low, dark = high
    linewidths=0.5,
    cbar_kws={"label": "F1-Score"},
)
plt.title("F1-Score Heatmap — max_features vs C (Exp5 pipeline)")
plt.xlabel("C (regularisation strength)")
plt.ylabel("max_features")
plt.tight_layout()
plt.show()

# ── Best combo ────────────────────────────────────────────────────────────────
best_tune = tune_df.iloc[0]
print(f"\nBest combination:")
print(f"  max_features = {int(best_tune['max_features']):,}")
print(f"  C            = {best_tune['C']}")
print(f"  Accuracy     = {best_tune['accuracy']:.4f}")
print(f"  F1-Score     = {best_tune['f1_score']:.4f}")
print(f"\nImprovement over original Exp5 (F1=0.8995):")
print(f"  Delta F1 = {best_tune['f1_score'] - 0.8995:+.4f}")
print(f"  Delta Acc = {best_tune['accuracy'] - 0.8985:+.4f}")

In [ ]:
# ── Effect of max_features (averaging across all C values) ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: F1 vs max_features for each C
for c_val in C_GRID:
    subset = tune_df[tune_df["C"] == c_val].sort_values("max_features")
    axes[0].plot(subset["max_features"], subset["f1_score"],
                 marker="o", label=f"C={c_val}")
axes[0].set_title("F1-Score vs max_features (per C)")
axes[0].set_xlabel("max_features")
axes[0].set_ylabel("F1-Score")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.4)

# Right: F1 vs C for each max_features
for max_f in MAX_FEATURES_GRID:
    subset = tune_df[tune_df["max_features"] == max_f].sort_values("C")
    axes[1].plot(subset["C"], subset["f1_score"],
                 marker="o", label=f"max_f={max_f:,}")
axes[1].set_title("F1-Score vs C (per max_features)")
axes[1].set_xlabel("C (regularisation)")
axes[1].set_ylabel("F1-Score")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.4)

plt.suptitle("Hyperparameter Sensitivity — Exp5 Pipeline", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# ── Quick summary of what each param does ─────────────────────────────────────
avg_by_maxf = tune_df.groupby("max_features")["f1_score"].mean().round(4)
avg_by_c    = tune_df.groupby("C")["f1_score"].mean().round(4)

print("Average F1 by max_features (across all C values):")
display(avg_by_maxf.to_frame())

print("\nAverage F1 by C (across all max_features values):")
display(avg_by_c.to_frame())

## Tuning Summary

The heatmap and line plots together answer two questions:

**Does increasing `max_features` help?**  
Generally yes - going from 5,000 to 20,000–50,000 features tends to improve F1 because the model has access to a richer vocabulary. However, gains plateau after a point.

**What `C` value works best?**  
A moderate-to-high C (e.g. 1.0–5.0) typically works well on IMDB. Very low C (0.1) over-regularises and loses performance; very high C (10.0) can slightly overfit.

The best combination from this grid is the one reported above - use those values as the final model configuration.

## Summary

We ran **5 experiments** to understand how preprocessing choices affect sentiment classification on the IMDB dataset.

### Key takeaways
- Even a **basic pipeline** (lowercase + tokenize + TF-IDF) gives strong results with Logistic Regression on IMDB  
- **Stopword removal** generally helps - it cuts down noise without losing much signal  
- **Stemming** can hurt or help depending on the dataset - overly aggressive stemming sometimes merges words that should stay distinct  
- **Bigrams** capture two-word phrases (e.g. *not good*) which unigrams miss — useful for negation handling  
- Logistic Regression is a solid baseline: fast, interpretable, and surprisingly competitive on high-dimensional text data

These results will be presented in the 2-page report and 5-slide deck.